# 🐱‍🔮 Sudoku Master — Entrenar CNN Solver (Colab + GPU)
Entrena la red que resuelve sudokus, con el CSV grande (~1.3 GB, millones de puzzles).
Optimizado para GPU de Colab. Guarda en Drive.

**Idea para la presentación:** comparar la solución de la red neuronal sola vs backtracking, midiendo el % de celdas que la red acierta sin ayuda.

## PASO 0 · Comprobar GPU

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('✓ GPU disponible:', gpus if gpus else '❌ NO — ve a Entorno de ejecución → Cambiar tipo → GPU')

from google.colab import drive
drive.mount('/content/drive')

## PASO 1 · Cargar el CSV grande por bloques
El CSV de 1.3 GB no cabe entero en RAM si lo procesas de golpe.
Cargamos N puzzles (ajusta MAX_PUZZLES según la RAM disponible).

In [ ]:
import pandas as pd
import numpy as np

RUTA_CSV = '/content/drive/MyDrive/Sudoku/sudoku.csv'
MAX_PUZZLES = 1_000_000   # sube o baja según RAM (1M ~ 8GB RAM al vectorizar)

# Detectar nombres de columnas (algunos CSV usan 'quizzes'/'solutions')
cabecera = pd.read_csv(RUTA_CSV, nrows=1)
cols = list(cabecera.columns)
print('Columnas del CSV:', cols)
col_puzzle   = 'puzzle'   if 'puzzle'   in cols else cols[0]
col_solucion = 'solution' if 'solution' in cols else cols[1]
print(f'Usando: puzzle="{col_puzzle}", solución="{col_solucion}"')

df = pd.read_csv(RUTA_CSV, nrows=MAX_PUZZLES)
print(f'✓ {len(df):,} puzzles cargados')

In [ ]:
# Vectorizar eficientemente: string de 81 chars → array (N, 81)
def a_array(serie):
    # Convierte la columna de strings a matriz numérica de forma vectorizada
    arr = np.frombuffer(''.join(serie.values).encode(), dtype=np.uint8) - ord('0')
    return arr.reshape(len(serie), 81)

X = a_array(df[col_puzzle]).astype('float32') / 9.0   # normalizado 0-1
y = a_array(df[col_solucion]).astype('int8') - 1       # clases 0-8

print(f'✓ X: {X.shape}  y: {y.shape}')
print(f'  Memoria X: {X.nbytes/1e6:.0f} MB, y: {y.nbytes/1e6:.0f} MB')

# Liberar el dataframe
del df
import gc; gc.collect()

## PASO 2 · Definir la CNN Solver
Misma arquitectura que ya tenías (Conv2D sobre el grid 9x9), que funciona bien.

In [ ]:
from tensorflow import keras

modelo_sudoku = keras.Sequential([
    keras.layers.Reshape((9, 9, 1), input_shape=(81,)),

    keras.layers.Conv2D(64,  (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(64,  (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(256, (3,3), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2D(256, (1,1), activation='relu', padding='same'),
    keras.layers.BatchNormalization(),

    keras.layers.Conv2D(9, (1,1), activation='softmax', padding='same'),
    keras.layers.Reshape((81, 9))
])

modelo_sudoku.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
modelo_sudoku.summary()

## PASO 3 · Entrenar
Con callbacks: baja el learning rate al estancarse y para si no mejora.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5, verbose=1),
    keras.callbacks.ModelCheckpoint('/content/drive/MyDrive/Sudoku/modelo_sudoku_mejor.keras',
                                    monitor='val_accuracy', save_best_only=True, verbose=1)
]

historia = modelo_sudoku.fit(
    X, y,
    epochs=30,
    batch_size=256,        # GPU permite batch grande → más rápido
    validation_split=0.1,
    callbacks=callbacks
)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(historia.history['accuracy'],     label='train')
axes[0].plot(historia.history['val_accuracy'], label='val')
axes[0].set_title('Accuracy por celda'); axes[0].legend()
axes[1].plot(historia.history['loss'],     label='train')
axes[1].plot(historia.history['val_loss'], label='val')
axes[1].set_title('Loss'); axes[1].legend()
plt.tight_layout(); plt.show()

acc = max(historia.history['val_accuracy'])
print(f'✓ Mejor accuracy por celda: {acc:.4f} ({acc*100:.2f}%)')

## PASO 4 · Evaluar de verdad: % de sudokus resueltos completos
La accuracy por celda engaña (98% por celda puede ser 0% de sudokus perfectos).
Lo que importa: ¿cuántos sudokus resuelve ENTEROS sin un solo fallo?

In [ ]:
# Tomar una muestra de validación
N_TEST = 2000
X_test = X[-N_TEST:]
y_test = y[-N_TEST:]

pred = modelo_sudoku.predict(X_test, batch_size=256, verbose=1)
pred_clases = np.argmax(pred, axis=2)   # (N, 81) con clases 0-8

# Accuracy por celda
acc_celda = np.mean(pred_clases == y_test)

# % de sudokus resueltos completos (todas las celdas correctas)
correctos_por_sudoku = np.all(pred_clases == y_test, axis=1)
acc_sudoku = np.mean(correctos_por_sudoku)

print(f'Accuracy por celda:    {acc_celda*100:.2f}%')
print(f'Sudokus 100% correctos: {acc_sudoku*100:.2f}%  ({correctos_por_sudoku.sum()}/{N_TEST})')
print('\n→ Para la presentación: la red sola resuelve perfecto el {:.1f}% de los sudokus.'.format(acc_sudoku*100))
print('  El backtracking corrige el resto garantizando el 100%.')

## PASO 5 · Guardar el modelo final

In [ ]:
modelo_sudoku.save('/content/drive/MyDrive/Sudoku/modelo_sudoku.keras')
print('✓ Modelo guardado en Drive: modelo_sudoku.keras')
print('  Descárgalo y ponlo en tu carpeta local modelos/')

## PASO 6 · Comparación CNN vs Backtracking (narrativa para la presentación)
Resuelve un sudoku de las tres formas y compara.

In [ ]:
import time, copy

def es_valido(t, fila, col, num):
    if num in t[fila]: return False
    if num in [t[i][col] for i in range(9)]: return False
    bf, bc = (fila//3)*3, (col//3)*3
    for i in range(bf, bf+3):
        for j in range(bc, bc+3):
            if t[i][j] == num: return False
    return True

def backtrack(t):
    for i in range(9):
        for j in range(9):
            if t[i][j] == 0:
                for num in range(1, 10):
                    if es_valido(t, i, j, num):
                        t[i][j] = num
                        if backtrack(t): return True
                        t[i][j] = 0
                return False
    return True

# Tomar un puzzle de test
idx = 0
puzzle = (X_test[idx]*9).astype(int).reshape(9,9)
solucion_real = (y_test[idx]+1).reshape(9,9)

# --- Método 1: CNN sola ---
t0 = time.time()
pred_cnn = np.argmax(modelo_sudoku.predict(X_test[idx:idx+1], verbose=0)[0], axis=1).reshape(9,9) + 1
# mantener pistas originales
sol_cnn = np.where(puzzle != 0, puzzle, pred_cnn)
t_cnn = time.time() - t0
aciertos_cnn = np.mean(sol_cnn == solucion_real) * 100

# --- Método 2: solo backtracking ---
t0 = time.time()
tablero_bt = puzzle.tolist()
backtrack(tablero_bt)
t_bt = time.time() - t0
aciertos_bt = np.mean(np.array(tablero_bt) == solucion_real) * 100

print('═'*45)
print(f'{"Método":<22}{"Aciertos":<12}{"Tiempo"}')
print('─'*45)
print(f'{"CNN (red neuronal)":<22}{aciertos_cnn:>6.1f}%     {t_cnn*1000:>7.1f} ms')
print(f'{"Backtracking":<22}{aciertos_bt:>6.1f}%     {t_bt*1000:>7.1f} ms')
print('═'*45)
print('\nLa CNN es instantánea pero puede fallar; el backtracking')
print('es 100% fiable. Combinándolos: rápido Y correcto.')